Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
import random
import math
from pathlib import Path
from typing import List, Tuple, Dict


In [ ]:
# Set style for plots
plt.style.use('ggplot')
sns.set_palette("husl")

# Paths
TRAIN_PATH = '/kaggle/input/conll003-englishversion/train.txt'
VALID_PATH = '/kaggle/input/conll003-englishversion/valid.txt'
TEST_PATH = '/kaggle/input/conll003-englishversion/test.txt'

# Load text data function
def load_text_data(file_path):
    """
    Load text data from various file formats
    """
    if file_path.endswith('.txt'):
        with open(file_path, 'r', encoding='utf-8') as file:
            lines = file.readlines()
    elif file_path.endswith('.csv'):
        df = pd.read_csv(file_path)
        lines = df['text'].tolist()
    else:
        raise ValueError("Unsupported file format")
    
    return [line.strip() for line in lines if line.strip()]

# Load and analyze data
print("=== TRAINING DATA ===")
train_text = load_text_data(TRAIN_PATH)
train_df = pd.DataFrame({'text': train_text})
print(f"Training dataset size: {len(train_df)} documents")

print("\n=== VALIDATION DATA ===")
valid_text = load_text_data(VALID_PATH)
valid_df = pd.DataFrame({'text': valid_text})
print(f"Validation dataset size: {len(valid_df)} documents")

print("\n=== TEST DATA ===")
test_text = load_text_data(TEST_PATH)
test_df = pd.DataFrame({'text': test_text})
print(f"Test dataset size: {len(test_df)} documents")


NLP approach

In [ ]:
!pip install seqeval transformers

### Code Summary
- Import PyTorch, Transformers, and SeqEval tools.  
- Define `set_seed` for reproducibility.  
- Run `set_seed(42)` to fix randomness.  


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from torch.optim import AdamW

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    get_linear_schedule_with_warmup,
)

from seqeval.metrics import classification_report, f1_score

# Reproducibility
def set_seed(seed: int = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

set_seed(42)

### Code Summary
- Define `read_conll(path)` to read a CoNLL-style file.  
- Extract tokens and NER tags sentence by sentence.  
- Skip empty lines and `-DOCSTART-`.  
- Return: `sentences` (list of tokens) and `ner_tags` (list of labels).  


In [ ]:
# Parser for CoNLL-style file
def read_conll(path: str) -> Tuple[List[List[str]], List[List[str]]]:
    """Read a CoNLL-style file and return sentences and corresponding NER tags."""
    sentences, ner_tags = [], []
    words, tags = [], []

    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                if words:
                    sentences.append(words)
                    ner_tags.append(tags)
                    words, tags = [], []
                continue
            
            # Skip document start markers
            if line.startswith('-DOCSTART-'):
                continue
            
            parts = line.split()
            if len(parts) >= 2:  # At least token and tag
                token = parts[0]
                tag = parts[-1]  # Last column is the tag
                words.append(token)
                tags.append(tag)
            else:
                # Handle malformed lines
                continue
    
    # Add the last sentence if exists
    if words:
        sentences.append(words)
        ner_tags.append(tags)
    
    return sentences, ner_tags


### Code Summary
- Load train, validation, and test data using `read_conll`.  
- Print number of sentences in each split.  
- Collect all unique tags from datasets.  
- Create mappings: `label2id` (tag → index) and `id2label` (index → tag).  
- Print total unique tags and list of tags.  


In [ ]:
# Load CoNLL data
print("Loading CoNLL data...")
train_sents, train_tags = read_conll(TRAIN_PATH)
valid_sents, valid_tags = read_conll(VALID_PATH)
test_sents, test_tags = read_conll(TEST_PATH)

print(f"Training: {len(train_sents)} sentences")
print(f"Validation: {len(valid_sents)} sentences")
print(f"Test: {len(test_sents)} sentences")

# Get all unique tags and create mappings
all_tags = set()
for tags in train_tags + valid_tags + test_tags:
    all_tags.update(tags)

all_tags = sorted(list(all_tags))
label2id = {tag: idx for idx, tag in enumerate(all_tags)}
id2label = {idx: tag for tag, idx in label2id.items()}

print(f"Unique tags: {len(all_tags)}")
print("Tags:", all_tags)

### Code Summary
- Set model config:  
  - `model_name`: BERT base (cased).  
  - `max_length`: 128 tokens.  
  - `batch_size`: 16.  
  - `learning_rate`: 2e-5.  
  - `num_epochs`: 3.  


In [ ]:
# Model configuration
model_name = "bert-base-cased"
max_length = 128
batch_size = 16
learning_rate = 2e-5
num_epochs = 3

### Code Summary
- Initialize tokenizer from pretrained BERT.  
- Define `NERDataset` class:  
  - Stores sentences, tags, tokenizer, and configs.  
  - `__len__`: returns dataset size.  
  - `__getitem__`:  
    - Tokenizes sentence.  
    - Aligns tokens with NER tags.  
    - Uses `-100` for padding/special tokens/subwords.  
    - Returns `input_ids`, `attention_mask`, and `labels`.  


In [ ]:
# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Custom Dataset class
class NERDataset(Dataset):
    def __init__(self, sentences, tags, tokenizer, label2id, max_length):
        self.sentences = sentences
        self.tags = tags
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.max_length = max_length
    
    def __len__(self):
        return len(self.sentences)
    
    def __getitem__(self, idx):
        sentence = self.sentences[idx]
        tags = self.tags[idx]
        
        # Tokenize
        encoding = self.tokenizer(
            sentence,
            is_split_into_words=True,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        # Create labels
        labels = []
        word_ids = encoding.word_ids()
        previous_word_idx = None
        
        for word_idx in word_ids:
            if word_idx is None:
                labels.append(-100)  # Special tokens
            elif word_idx != previous_word_idx:
                labels.append(self.label2id[tags[word_idx]])
            else:
                labels.append(-100)  # Subword tokens
            previous_word_idx = word_idx
        
        # Pad labels to max_length
        while len(labels) < self.max_length:
            labels.append(-100)
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(labels[:self.max_length])
        }

### Code Summary
- Create train, validation, and test datasets using `NERDataset`.  
- Wrap them in `DataLoader` for batching/shuffling.  
- Set device (GPU if available, else CPU).  
- Load pretrained BERT for token classification with tag mappings.  
- Move model to device.  


In [ ]:
# Create datasets and dataloaders
train_dataset = NERDataset(train_sents, train_tags, tokenizer, label2id, max_length)
valid_dataset = NERDataset(valid_sents, valid_tags, tokenizer, label2id, max_length)
test_dataset = NERDataset(test_sents, test_tags, tokenizer, label2id, max_length)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Initialize model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(all_tags),
    id2label=id2label,
    label2id=label2id
)
model.to(device)

# Optimizer and scheduler
optimizer = AdamW(model.parameters(), lr=learning_rate)
total_steps = len(train_loader) * num_epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

# Training function
def train_epoch(model, dataloader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    
    for batch in dataloader:
        optimizer.zero_grad()
        
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
    
    return total_loss / len(dataloader)

In [ ]:
# Evaluation function
def evaluate(model, dataloader, device):
    model.eval()
    total_loss = 0
    predictions = []
    true_labels = []
    
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            
            total_loss += outputs.loss.item()
            
            # Get predictions
            logits = outputs.logits
            preds = torch.argmax(logits, dim=-1)
            
            # Remove padding and special tokens
            for i in range(len(preds)):
                mask = labels[i] != -100
                valid_preds = preds[i][mask].cpu().numpy()
                valid_labels = labels[i][mask].cpu().numpy()
                
                # Convert to string labels
                pred_labels = [id2label[p] for p in valid_preds]
                true_label_strs = [id2label[l] for l in valid_labels]
                
                predictions.append(pred_labels)
                true_labels.append(true_label_strs)
    
    avg_loss = total_loss / len(dataloader)
    f1 = f1_score(true_labels, predictions)
    
    return avg_loss, f1, predictions, true_labels

In [ ]:
# Training loop
best_val_f1 = 0
save_dir = '/kaggle/working/best_model'

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch + 1}/{num_epochs}")
    
    # Training
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
    print(f"Training loss: {train_loss:.4f}")
    
    # Validation
    val_loss, val_f1, _, _ = evaluate(model, valid_loader, device)
    print(f"Validation loss: {val_loss:.4f}, F1: {val_f1:.4f}")
    
    # Save best model
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        model.save_pretrained(save_dir)
        tokenizer.save_pretrained(save_dir)
        print(f"Saved best model with F1: {val_f1:.4f}")

# Load best model and test
print("\n=== TESTING ===")
model = AutoModelForTokenClassification.from_pretrained(save_dir)
model.to(device)

test_loss, test_f1, test_preds, test_true = evaluate(model, test_loader, device)
print(f"Test loss: {test_loss:.4f}, F1: {test_f1:.4f}")

# Detailed classification report
print("\nClassification Report:")
print(classification_report(test_true, test_preds))

# Save predictions
def save_predictions(sentences, predictions, output_path):
    with open(output_path, 'w', encoding='utf-8') as f:
        for sent, preds in zip(sentences, predictions):
            for token, pred in zip(sent, preds):
                f.write(f"{token} {pred}\n")
            f.write('\n')

save_predictions(test_sents, test_preds, '/kaggle/working/test_predictions.txt')
print("Predictions saved to '/kaggle/working/test_predictions.txt'")

In [ ]:
# Calculate exact accuracy
def calculate_accuracy(true_labels, predictions):
    correct = 0
    total = 0
    
    for true_seq, pred_seq in zip(true_labels, predictions):
        for true_label, pred_label in zip(true_seq, pred_seq):
            if true_label != 'O':  # Only count actual entities, not 'O' tags
                total += 1
                if true_label == pred_label:
                    correct += 1
    
    return correct / total if total > 0 else 0

accuracy = calculate_accuracy(test_true, test_preds)
print(f"Exact Accuracy (on entities only): {accuracy:.4f} ({accuracy*100:.2f}%)")

In [ ]:
# INFERENCE ON UNSEEN DATA
print("=== RUNNING INFERENCE ON UNSEEN DATA ===")

# Load your saved best model (if not already loaded)
model = AutoModelForTokenClassification.from_pretrained(save_dir)
tokenizer = AutoTokenizer.from_pretrained(save_dir)
model.to(device)
model.eval()

def predict_ner(text, model, tokenizer, id2label, max_length=128):
    """
    Predict NER tags for a single text input
    """
    # Tokenize the text
    tokens = text.split()  # Simple whitespace tokenization
    encoding = tokenizer(
        tokens,
        is_split_into_words=True,
        return_tensors='pt',
        truncation=True,
        max_length=max_length,
        padding='max_length'
    )
    
    # Move to device
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    
    # Predict
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        predictions = torch.argmax(logits, dim=-1)
    
    # Convert to labels
    word_ids = encoding.word_ids()
    predicted_labels = []
    previous_word_idx = None
    
    for idx, word_idx in enumerate(word_ids):
        if word_idx is None:
            continue
        if word_idx != previous_word_idx:
            predicted_labels.append(id2label[predictions[0][idx].item()])
        previous_word_idx = word_idx
    
    return tokens, predicted_labels

def pretty_print_ner(tokens, labels):
    """
    Pretty print the NER results with color coding
    """
    from termcolor import colored
    
    print("NER Results:")
    print("-" * 50)
    
    for token, label in zip(tokens, labels):
        if label == 'O':
            print(token, end=' ')
        else:
            entity_type = label.split('-')[-1]  # Get B-PER → PER
            color_map = {
                'PER': 'red',
                'ORG': 'green', 
                'LOC': 'blue',
                'MISC': 'yellow'
            }
            color = color_map.get(entity_type, 'white')
            print(colored(f"{token}({entity_type})", color), end=' ')
    print("\n" + "-" * 50)

# Example sentences for inference
test_sentences = [
    "Apple Inc. is headquartered in Cupertino, California and was founded by Steve Jobs.",
    "Barack Obama visited Paris last month to meet with Emmanuel Macron.",
    "The United Nations is located in New York City and works on global issues.",
    "Microsoft Corporation, led by Satya Nadella, is based in Redmond, Washington.",
    "Elon Musk founded Tesla and SpaceX, both innovative companies in California."
]

print("Running inference on example sentences...\n")

for i, sentence in enumerate(test_sentences, 1):
    print(f"Example {i}:")
    print(f"Input: {sentence}")
    
    tokens, predictions = predict_ner(sentence, model, tokenizer, id2label)
    pretty_print_ner(tokens, predictions)
    print()

# Interactive inference
print("=== INTERACTIVE INFERENCE ===")
print("Type your own sentences for NER prediction (type 'quit' to exit):")

while True:
    user_input = input("\nEnter a sentence: ").strip()
    
    if user_input.lower() in ['quit', 'exit', 'q']:
        print("Exiting inference mode...")
        break
    
    if not user_input:
        continue
    
    try:
        tokens, predictions = predict_ner(user_input, model, tokenizer, id2label)
        pretty_print_ner(tokens, predictions)
        
        # Also show structured output
        print("\nStructured Entities Found:")
        current_entity = None
        current_text = []
        
        for token, label in zip(tokens, predictions):
            if label.startswith('B-'):
                # Save previous entity if exists
                if current_entity:
                    print(f"  {current_entity}: {' '.join(current_text)}")
                
                # Start new entity
                current_entity = label[2:]  # Remove B-
                current_text = [token]
                
            elif label.startswith('I-') and current_entity == label[2:]:
                current_text.append(token)
            else:
                # Save previous entity if exists
                if current_entity:
                    print(f"  {current_entity}: {' '.join(current_text)}")
                    current_entity = None
                    current_text = []
        
        # Save the last entity
        if current_entity:
            print(f"  {current_entity}: {' '.join(current_text)}")
            
    except Exception as e:
        print(f"Error processing input: {e}")

print("\nInference completed!")